# Supermarket Case Study: Cleaning and Branch Performance
One pipeline from the raw file to a management summary: load, audit, clean, engineer time features, compare against targets, find busy times, chart and export.

## 1. Load and First Look

In [1]:
import pandas as pd
import numpy as np


In [7]:
df=pd.read_csv("supermarket_sales_case_study.csv")
print(df.shape)
df.head()
df.info()


(8631, 17)
<class 'pandas.DataFrame'>
RangeIndex: 8631 entries, 0 to 8630
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Invoice ID               8631 non-null   str    
 1   Branch                   8631 non-null   str    
 2   City                     8631 non-null   str    
 3   Customer type            8567 non-null   str    
 4   Gender                   8631 non-null   str    
 5   Product line             8631 non-null   str    
 6   Unit price               8631 non-null   float64
 7   Quantity                 8631 non-null   int64  
 8   Tax 5%                   8631 non-null   float64
 9   Total                    8631 non-null   float64
 10  Date                     8614 non-null   str    
 11  Time                     8631 non-null   str    
 12  Payment                  8631 non-null   str    
 13  cogs                     8631 non-null   float64
 14  gross margin percentage 

## 2. Audit Before Fixing

In [18]:
audit=pd.DataFrame({
    "Missing_count" : df.isnull().sum(),
    "Missing_percent" : (df.isnull().mean()*100).round(2)
})
print(audit[audit.Missing_count >0])
print("Duplicate Rows :",df.duplicated().sum())

               Missing_count  Missing_percent
Customer type             64             0.74
Date                      17             0.20
Rating                    88             1.02
Duplicate Rows : 30


In [ ]:
rows_in = len(df)
trail = [{"step": "Raw file loaded", "rows": rows_in, "removed": 0}]

def log(step):
    prev = trail[-1]["rows"]
    trail.append({"step": step, "rows": len(df), "removed": prev - len(df)})

print("Rows sharing an Invoice ID:", df.duplicated("Invoice ID", keep=False).sum())
print("Gender values:", df["Gender"].unique())
print("Payment values:", df["Payment"].unique())

## 3. Drop Exact Duplicates

In [20]:
df=df.drop_duplicates()
print("(After cleaning Rows) Duplicate count :",df.duplicated().sum())

(After cleaning Rows) Duplicate count : 0


In [ ]:
log("Exact duplicates dropped")

## 4. Resolve Conflicting Invoice IDs
**Rule:** when an Invoice ID appears more than once with different values, the later row is a correction, so the last row is kept.

In [ ]:
df = df.drop_duplicates(subset="Invoice ID", keep="last")
log("Conflicting Invoice IDs resolved")

## 5. Standardise Text Columns

In [ ]:
gender_map = {name.lower(): name for name in ["Male", "Female"]}
pay_map = {name.lower(): name for name in ["Cash", "Credit card", "Ewallet"]}

df["Gender"] = df["Gender"].str.split().str.join(" ").str.lower().map(gender_map)
df["Payment"] = df["Payment"].str.split().str.join(" ").str.lower().map(pay_map)

print("Unmapped Gender:", df["Gender"].isna().sum())
print("Unmapped Payment:", df["Payment"].isna().sum())
print(df["Gender"].unique(), df["Payment"].unique())
log("Text columns standardised")

## 6. Fix Negative Quantities

In [ ]:
print("Negative quantities:", (df["Quantity"] < 0).sum())
df["Quantity"] = df["Quantity"].abs()
log("Negative quantities fixed")

## 7. Parse Mixed Dates
First month/day/year, then day-month-year; anything else becomes missing.

In [ ]:
date_text = df["Date"].str.strip()
first_try = pd.to_datetime(date_text, format="%m/%d/%Y", errors="coerce")
second_try = pd.to_datetime(date_text, format="%d-%m-%Y", errors="coerce")
df["Date"] = first_try.fillna(second_try)

print("Dates missing after parsing:", df["Date"].isna().sum())
log("Dates parsed")

## 8. Check Total

In [ ]:
calc_total = df["Quantity"] * df["Unit price"] * 1.05
off_rows = (df["Total"] - calc_total).abs() > 0.50

print("Totals replaced:", off_rows.sum())
df.loc[off_rows, "Total"] = calc_total[off_rows]
log("Totals checked")

## 9. Handle What Is Still Missing

In [ ]:
df["Customer type"] = df["Customer type"].fillna("Unknown")
df["Rating"] = df["Rating"].fillna(df["Rating"].median())
df = df.dropna(subset=["Date"]).reset_index(drop=True)
log("Missing values handled, undated rows dropped")

## 10. Audit Trail

In [ ]:
pd.DataFrame(trail)

## 11. Time Columns

In [ ]:
df["Year"] = df["Date"].dt.year
df["Quarter"] = df["Date"].dt.year.astype(str) + "Q" + df["Date"].dt.quarter.astype(str)
df["Day_of_week"] = df["Date"].dt.day_name()
df["Hour"] = df["Time"].str.strip().str.split(":").str[0].astype(int)
df[["Date", "Year", "Quarter", "Day_of_week", "Hour"]].head()

## 12. Bucket the Trading Day

In [ ]:
conditions = [
    (df["Hour"] >= 10) & (df["Hour"] < 13),
    (df["Hour"] >= 13) & (df["Hour"] < 17),
    (df["Hour"] >= 17) & (df["Hour"] < 21),
]
choices = ["Morning", "Afternoon", "Evening"]
df["Bucket"] = np.select(conditions, choices, default="Other")

bucket_table = df["Bucket"].value_counts().rename("Transactions").to_frame()
bucket_table["Hours"] = pd.Series({"Morning": 3, "Afternoon": 4, "Evening": 4})
bucket_table = bucket_table.dropna()
bucket_table["Per_hour"] = (bucket_table["Transactions"] / bucket_table["Hours"]).round(1)
bucket_table

## 13. Compare Against Target

In [ ]:
rev = df.groupby(["Branch", "Quarter"])["Total"].sum().round(2).reset_index(name="Revenue")

targets = pd.DataFrame({"Branch": sorted(df["Branch"].unique())})
targets["Target"] = 100000

rev = rev.merge(targets, on="Branch", how="left")
rev["Variance"] = rev["Revenue"] - rev["Target"]
rev["Pct_of_target"] = (rev["Revenue"] / rev["Target"] * 100).round(1)
rev["Hit"] = rev["Revenue"] >= rev["Target"]
rev

## 14. Summarise Each Branch

In [ ]:
summary = rev.groupby("Branch").agg(
    Avg_quarterly=("Revenue", "mean"),
    Quarters_hit=("Hit", "sum"),
    Best_revenue=("Revenue", "max"),
    Worst_revenue=("Revenue", "min"),
).round(2)

summary["Best_quarter"] = rev.loc[rev.groupby("Branch")["Revenue"].idxmax()].set_index("Branch")["Quarter"]
summary["Worst_quarter"] = rev.loc[rev.groupby("Branch")["Revenue"].idxmin()].set_index("Branch")["Quarter"]

yearly = df.pivot_table(values="Total", index="Branch", columns="Year", aggfunc="sum")
summary["YoY_growth_pct"] = ((yearly[2024] / yearly[2023] - 1) * 100).round(2)
summary

## 15. Find the Busy Times

In [ ]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

calendar = pd.date_range(df["Date"].min(), df["Date"].max())
weekday_days = pd.Series(calendar.day_name()).value_counts()

day_counts = df["Day_of_week"].value_counts()
avg_per_day = (day_counts / weekday_days).reindex(day_order).round(1)
avg_per_day

In [ ]:
cell_counts = df.pivot_table(values="Invoice ID", index="Hour", columns="Day_of_week", aggfunc="count")
heat_data = (cell_counts / weekday_days).reindex(columns=day_order).fillna(0)
heat_data.round(2)

## 16. Charts

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

In [ ]:
plt.figure(figsize=(11, 4))
sns.heatmap(heat_data, cmap="YlOrRd", linewidths=0.3, annot=True, fmt=".1f")
plt.title("Average Transactions per Day: Hour x Weekday")
plt.tight_layout()
plt.show()

In [ ]:
rev_wide = rev.pivot_table(values="Revenue", index="Quarter", columns="Branch")

fig, ax = plt.subplots(figsize=(10, 5))
for branch_name in rev_wide.columns:
    ax.plot(rev_wide.index, rev_wide[branch_name], marker="o", label="Branch " + branch_name)
ax.axhline(100000, color="firebrick", linestyle="--", label="Target")
ax.set_title("Quarterly Revenue by Branch vs Target")
ax.set_ylabel("Revenue ($)")
ax.legend()
plt.tight_layout()
plt.show()

## 17. Export the Clean Data

In [ ]:
df.to_csv("supermarket_clean.csv", index=False)
print(f"wrote {len(df):,} clean rows")

## 18. Management Summary

In [ ]:
top_branch = summary["Avg_quarterly"].idxmax()
fast_branch = summary["YoY_growth_pct"].idxmax()
busy_day = avg_per_day.idxmax()
busy_bucket = bucket_table["Per_hour"].idxmax()

print(f"1. {rows_in:,} rows loaded; {len(df):,} kept after cleaning ({len(df) / rows_in * 100:.1f}%).")
print(f"2. Total clean revenue is ${df['Total'].sum():,.2f} across {df['Branch'].nunique()} branches.")
print(f"3. Branch {top_branch} leads with ${summary.loc[top_branch, 'Avg_quarterly']:,.2f} average quarterly revenue.")
print(f"4. Targets were hit in {int(summary['Quarters_hit'].sum())} of {len(rev)} branch-quarters; Branch {fast_branch} grew fastest year on year ({summary.loc[fast_branch, 'YoY_growth_pct']:.2f}%).")
print(f"5. {busy_day} is the busiest weekday and the {busy_bucket.lower()} is the busiest part of the day.")